# Notebook 02 — Landing Zone CSV → Delta Lake (Bronze)

Lê os **11 arquivos CSV** do bucket `landing-zone` e converte cada um para o formato **Delta Lake** no bucket `bronze`.

**Fluxo:**
```
MinIO s3a://landing-zone/{tabela}.csv
    ──Spark Read──► DataFrame
    ──Delta Write──► MinIO s3a://bronze/{tabela}/
```

## 1. Imports e configuração

In [1]:
import os
import boto3
from botocore.client import Config
from dotenv import load_dotenv

load_dotenv()

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT', 'http://localhost:9020')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY', 'minioadmin')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET', 'landing-zone')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET', 'bronze')
S3A_ENDPOINT     = MINIO_ENDPOINT.replace('http://', '').replace('https://', '')

# DEVE ser definido ANTES de importar pyspark
os.environ['PYSPARK_SUBMIT_ARGS'] = (
    '--packages '
    'io.delta:delta-spark_2.12:3.2.0,'
    'org.apache.hadoop:hadoop-aws:3.3.4,'
    'com.amazonaws:aws-java-sdk-bundle:1.12.262 '
    'pyspark-shell'
)

print(f'Landing Zone: s3a://{LANDING_BUCKET}')
print(f'Bronze:       s3a://{BRONZE_BUCKET}')
print('PYSPARK_SUBMIT_ARGS configurado!')

Landing Zone: s3a://landing-zone
Bronze:       s3a://bronze
PYSPARK_SUBMIT_ARGS configurado!


## 2. Criar SparkSession com Delta Lake + MinIO (S3A)

In [2]:
from pyspark.sql import SparkSession
from delta import DeltaTable

spark = (
    SparkSession.builder
    .appName('landing-to-bronze-delta')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.hadoop.fs.s3a.endpoint', f'http://{S3A_ENDPOINT}')
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)

spark.sparkContext.setLogLevel('WARN')
print(f'Spark {spark.version} iniciado!')

26/05/06 23:03:29 WARN Utils: Your hostname, LAPTOP-NGVDQKNB resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/06 23:03:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/mnt/c/Users/mazuc/OneDrive/%c3%81rea%20de%20Trabalho/SATC/5%c2%b0%20FASE/Engenharia%20de%20Dados/trabalho-2-delta-minio/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thiago/.ivy2/cache
The jars for the packages stored in: /home/thiago/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-7c851f45-27cd-40cf-ae8c-0f5a390ae1e1;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/hadoop/hadoop-aws/3.3.4/hadoop-aws-3.3.4.jar ...
	[SUCCESSFUL ] org.apache.hadoop#hadoop-aws;3.3.4!hadoop-aws.jar (225ms)
downloading https://repo1.maven.org/maven2/com/amazonaws/aws-java-sdk-bundle/1.12.262/aws-java-sdk-bundle-

Spark 3.5.3 iniciado!


## 3. Criar bucket bronze no MinIO

In [3]:
s3 = boto3.client(
    's3',
    endpoint_url=MINIO_ENDPOINT,
    aws_access_key_id=MINIO_ACCESS_KEY,
    aws_secret_access_key=MINIO_SECRET_KEY,
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

existing = [b['Name'] for b in s3.list_buckets()['Buckets']]
if BRONZE_BUCKET not in existing:
    s3.create_bucket(Bucket=BRONZE_BUCKET)
    print(f'Bucket criado: {BRONZE_BUCKET}')
else:
    print(f'Bucket já existe: {BRONZE_BUCKET}')

Bucket já existe: bronze


## 4. Converter cada CSV para Delta Lake

In [4]:
# Lista os CSVs disponíveis no landing-zone
response = s3.list_objects_v2(Bucket=LANDING_BUCKET)
csvs = [obj['Key'] for obj in response.get('Contents', []) if obj['Key'].endswith('.csv')]

print(f'{len(csvs)} arquivos CSV encontrados no landing-zone:\n')

resultados = []

for csv_key in csvs:
    tabela = csv_key.replace('.csv', '')
    landing_path = f's3a://{LANDING_BUCKET}/{csv_key}'
    bronze_path  = f's3a://{BRONZE_BUCKET}/{tabela}'

    # Lê o CSV
    df = spark.read.csv(landing_path, header=True, inferSchema=True)

    # Escreve como Delta Lake (overwrite para re-execuções)
    df.write.format('delta').mode('overwrite').save(bronze_path)

    count = df.count()
    resultados.append({'tabela': tabela, 'registros': count, 'path': bronze_path})
    print(f'  {tabela:15s} → {count:5d} registros → {bronze_path}')

print(f'\nTotal: {sum(r["registros"] for r in resultados)} registros convertidos para Delta Lake')

11 arquivos CSV encontrados no landing-zone:



26/05/06 23:05:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

  apolice         →    30 registros → s3a://bronze/apolice


  carro           →    50 registros → s3a://bronze/carro
  cliente         →    60 registros → s3a://bronze/cliente
  endereco        →    60 registros → s3a://bronze/endereco
  estado          →    27 registros → s3a://bronze/estado
  marca           →    10 registros → s3a://bronze/marca
  modelo          →    20 registros → s3a://bronze/modelo
  municipio       →    40 registros → s3a://bronze/municipio
  regiao          →     5 registros → s3a://bronze/regiao
  sinistro        →    20 registros → s3a://bronze/sinistro
  telefone        →    50 registros → s3a://bronze/telefone

Total: 372 registros convertidos para Delta Lake


## 5. Validação — verificar tabelas Delta

In [5]:
print('=== Validação das tabelas Delta no bucket bronze ===')
for r in resultados:
    tabela = r['tabela']
    path   = r['path']

    is_delta = DeltaTable.isDeltaTable(spark, path)
    count    = spark.read.format('delta').load(path).count()

    status = 'OK' if is_delta and count == r['registros'] else 'ERRO'
    print(f'  [{status}] {tabela:15s} | isDelta={is_delta} | {count} registros')

=== Validação das tabelas Delta no bucket bronze ===


26/05/06 23:06:29 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

  [OK] apolice         | isDelta=True | 30 registros


  [OK] carro           | isDelta=True | 50 registros


  [OK] cliente         | isDelta=True | 60 registros


...

  [OK] modelo          | isDelta=True | 20 registros


  [OK] municipio       | isDelta=True | 40 registros


  [OK] regiao          | isDelta=True | 5 registros


  [OK] sinistro        | isDelta=True | 20 registros


  [OK] telefone        | isDelta=True | 50 registros


## 6. Inspecionar schema e dados de uma tabela

In [6]:
# Exemplo com a tabela apolice
df_apolice = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/apolice')

print('Schema da tabela apolice:')
df_apolice.printSchema()

print('\nPrimeiros registros:')
df_apolice.show(5, truncate=False)

Schema da tabela apolice:
root
 |-- id: integer (nullable = true)
 |-- cliente_id: integer (nullable = true)
 |-- carro_id: integer (nullable = true)
 |-- numero_apolice: string (nullable = true)
 |-- data_inicio: date (nullable = true)
 |-- data_fim: date (nullable = true)
 |-- tipo_cobertura: string (nullable = true)
 |-- valor_premio: double (nullable = true)
 |-- valor_cobertura: double (nullable = true)
 |-- status: string (nullable = true)


Primeiros registros:


+---+----------+--------+--------------+-----------+----------+--------------+------------+---------------+------+
|id |cliente_id|carro_id|numero_apolice|data_inicio|data_fim  |tipo_cobertura|valor_premio|valor_cobertura|status|
+---+----------+--------+--------------+-----------+----------+--------------+------------+---------------+------+
|1  |1         |1       |APL-2023-0001 |2023-01-15 |2024-01-15|Completo      |2800.0      |65000.0        |ativa |
|2  |2         |2       |APL-2023-0002 |2023-02-20 |2024-02-20|Terceiros     |980.0       |30000.0        |ativa |
|3  |3         |3       |APL-2023-0003 |2023-03-10 |2024-03-10|Completo      |4200.0      |115000.0       |ativa |
|4  |4         |4       |APL-2023-0004 |2023-04-05 |2024-04-05|Completo      |3100.0      |82000.0        |ativa |
|5  |5         |5       |APL-2023-0005 |2023-05-18 |2024-05-18|Terceiros     |1200.0      |30000.0        |ativa |
+---+----------+--------+--------------+-----------+----------+--------------+--